# 💧 LFM2 Inference with Ollama

This notebook demonstrates how to use the [Ollama](https://ollama.com) API to run [LFM2](https://huggingface.co/collections/LiquidAI/lfm2-67d775f3b4b6fe79fbb21bda) and [LFM2.5](https://huggingface.co/collections/LiquidAI/lfm25-6839e3e26b2a9fdbde95b341) models.

> ⚠️ **Note:** Ollama is intended to run locally on your machine. This notebook shows the Python and curl API usage to get Ollama running in Colab. Install Ollama from [ollama.com/download](https://ollama.com/download) and follow the [Liquid Docs](https://docs.liquid.ai/docs/inference/ollama) to get started. Also, right now LFM VL models are currently not working with ollama, we have an [open PR](https://github.com/ollama/ollama/pull/14069) to resolve this quickly.

## Installation

In [ ]:
# Colab specific settings
# !modal_skip
!sudo apt install zstd
!sudo apt update
!sudo apt install -y pciutils

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## Starting the Server

On most dedicated machines you won't have to manually run `ollama serve`, but in Colab it doesn't start automatically.

In [ ]:
import subprocess, time

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Give the server a moment to bind to the port
time.sleep(3)

print("Ollama server started")

Pull the model GGUF from Huggingface to run via the server endpoint.

In [ ]:
model = "hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF"

subprocess.run(
    ["ollama", "pull", model],
    check=True,
)

## Python Client (OpenAI-compatible)

Use the OpenAI Python client to interact with Ollama:

In [ ]:
!uv pip install openai -q

In [ ]:
from openai import OpenAI

# Connect to local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="not-needed"
)

# Chat completion
response = client.chat.completions.create(
    model="hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF",
    messages=[
        {"role": "user", "content": "What is C. elegans?"}
    ],
    temperature=0.7,
    max_tokens=512
)
print(response.choices[0].message.content)

## Streaming Responses

In [ ]:
stream = client.chat.completions.create(
    model="hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF",
    messages=[
        {"role": "user", "content": "Tell me a story about space exploration."}
    ],
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

## Curl Examples

You can also use curl to interact with the Ollama API:

In [ ]:
# Chat API
# !modal_skip_rest
%%bash
curl -s http://localhost:11434/api/chat -d '{
  "model": "hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF",
  "messages": [{"role": "user", "content": "What is machine learning?"}],
  "stream": false
}' | python -c "import sys, json; print(json.load(sys.stdin)['message']['content'])"

In [ ]:
# Generate API (simple completion)
%%bash
curl -s http://localhost:11434/api/generate -d '{
  "model": "hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF",
  "prompt": "What is artificial intelligence?",
  "stream": false
}' | python -c "import sys, json; print(json.load(sys.stdin)['response'])"

## Resources

- [LFM2 Documentation](https://docs.liquid.ai/docs/inference/ollama)
- [LFM2 GGUF Models on Hugging Face](https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF)
- [Ollama Documentation](https://ollama.com)